# Prompt Diary

First, I started by explaining my reasoning.

I began by analyzing the Excel file as a whole. I noticed that it contained several columns that were not necessary for calculating Recency, Frequency, and Monetary values. Therefore, I decided that after performing the analysis, I would create a separate dataset containing only the CustomerID and the three RFM features.

To clean the main dataset, I first used the dropna() function to remove all rows where the CustomerID was missing.

Next, I filtered the dataset to keep only rows where both Quantity and UnitPrice were positive.

While reading the dataset description, I also noticed that some InvoiceNo values could start with the letter "C", which indicates that the transaction was cancelled. Therefore, I removed those rows as well.

After these cleaning steps, I performed a simple verification to ensure the dataset was properly cleaned by checking that there were no missing CustomerID values and no negative quantities.



# RFM Calculation

After cleaning the dataset, I moved on to implementing the three functions required to calculate Recency, Frequency, and Monetary.

###  Recency

For the Recency calculation, I used pd.to_datetime() to convert the InvoiceDate column into a pandas datetime format. This function converts arrays or Series into pandas datetime objects.

I used the parameter dayfirst=True because in the Excel file the day appears before the month.

Next, I defined today_date as the most recent date in the dataset using .max(). This value acts as the reference point for calculating how many days have passed since each customer's last purchase.

To determine the most recent purchase of each customer, I used groupby("CustomerID") and calculated the maximum purchase date using .max().

The Recency value was then calculated by subtracting the customer's last purchase date from the dataset's most recent date.

Finally, I created a DataFrame containing CustomerID and Recency, which would later be merged with the other RFM metrics.

### Frequency

For the Frequency function, I used a similar approach with groupby.

One important observation was that the original dataset is organized by items, not by transactions. This means that a single purchase can appear in multiple rows if multiple items were bought.

Therefore, instead of simply counting rows, I used .nunique() on InvoiceNo, which represents a transaction identifier. This allowed me to count the number of distinct purchases made by each customer.

As with Recency, I created a separate DataFrame containing CustomerID and Frequency.

### Monetary

For the Monetary calculation, I first created a new column called TotalPrice, which represents the total value of each line:

TotalPrice = Quantity * UnitPrice

Then I grouped the dataset by CustomerID and summed the TotalPrice values to obtain the total amount spent by each customer.

This produced a DataFrame containing CustomerID and Monetary.

### Merging the RFM Features

After calculating the three metrics, I merged the three DataFrames into a single dataset containing:

- CustomerID

- Recency

- Frequency

- Monetary

While inspecting the results, I noticed that some customers had a Monetary value of 0. These customers likely returned everything they purchased.

Other possible explanations could be UnitPrice = 0 or cancelled orders, but those cases were already removed during the cleaning step.

Since these customers generated no revenue for the company, I decided to remove them using a function called clean_rfm_data, keeping only customers with Monetary > 0.

# Business Insights

Finally, I created two functions to identify customer segments for the marketing team: Champions and At-Risk High Spenders.

### Champions

To identify the 50 best customers, the most important feature is Monetary, since these customers generate the most revenue.

Therefore, I sorted the dataset prioritizing:

highest Monetary value

highest Frequency

lowest Recency

The sorting was implemented using the ascending parameter so that Monetary and Frequency are sorted in descending order and Recency in ascending order.

The top 50 customers from this ranking were selected as Champions.

### At-Risk High Spenders

For the At-Risk High Spenders, I needed to identify customers who historically spent a lot but have not purchased recently.

Initially, I attempted to use the same approach as for Champions, but I noticed that the resulting Monetary values were too low.

Therefore, I introduced a threshold to define high spenders.

I selected the top 20% of customers based on Monetary value, and from this subset I identified the 50 customers with the highest Recency values (customers who have not purchased recently) and the lowest Frequency values.

These customers were classified as At-Risk High Spenders, since they used to be valuable but may now be disengaging.

# Output

Finally, as requested in the challenge, I exported two CSV files:

champions.csv containing the Customer IDs of the top 50 Champions

at_risk_high_spenders.csv containing the Customer IDs of the 50 At-Risk High Spenders